In [1]:
import pandas as pd
import snowflake.connector
import json
import re

In [2]:
conn = snowflake.connector.connect(
    account="VSB79059-HJB86910",
    user="TYLER.HAAS@INNOVATION.CA.GOV",
    authenticator="externalbrowser",
    role="TRANSFORMER_ENGCA_DEV",
)

cur = conn.cursor()
cur.execute("""
    SELECT
        *
    FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS s
    WHERE PUBLICATION_STATUS = 'published'
""")

df = cur.fetch_pandas_all()
df.reset_index(names="temp_id", inplace=True)

In [3]:
df.head()

,temp_id,SURVEY_RESPONDENT_ID,SURVEY_ID,AGE,GENDER_ARRAY,GENDER_CATEGORY,RACE_ETHNICITY_ARRAY,RACE_ETHNICITY_CATEGORY,USER_STATUS,PUBLICATION_STATUS,...,ROLE_AT_WORK,COUNTY,REGION,FIELD_OF_WORK,ECONOMIC_IMPACT_EXPECTATION,GOVERNMENT_ACTION_SUGGESTION,PERSONAL_AI_IMPACT,AVAILABILITY_FOR_DISCUSSION,FIELDS_COMPLETED_COUNT,_LOADED_AT
0,0,b2768fb1-bcc7-4350-875c-ec9f44a6c29d,27bcca43-d971-45c6-8e40-ebce4a486b5b,25-44,"[\n ""Woman""\n]",Woman (only),"[\n ""Hispanic or Latino""\n]",Hispanic or Latino (only),active,published,...,Employee (non-management),Los Angeles County,Los Angeles,Manufacturing,It would be like the movie idiocracy. A societ...,Narrowly define what AI can and cannot do.,There is an expectation that AI would be runni...,No,11,2026-05-27 10:40:53.195014-07:00
1,1,755de677-19c0-4522-a92b-c82472c3cd12,c9c1a994-20ff-4977-9b50-4de2187695eb,25-44,"[\n ""Man""\n]",Man (only),"[\n ""Hispanic or Latino""\n]",Hispanic or Latino (only),active,published,...,I don't currently work,Tulare County,Central Valley,"Arts, entertainment, or media",AI will make the economy worse. Nobody is actu...,Tax and regulate generative AI and LLM. They h...,Yes. AI has made the job market more difficult.,Maybe,11,2026-05-27 10:40:53.195014-07:00
2,2,71ccc7b9-9496-4327-bc78-39f32f48347e,016b0d8f-fc98-4894-9ac4-32fa895c861e,25-44,"[\n ""Man""\n]",Man (only),"[\n ""White""\n]",White (only),active,published,...,Employee (non-management),San Francisco County,Bay Area,Information technology,AI will almost certainly lead to massive unemp...,The government should nationalize critical ind...,Engineers have become much more productive at ...,Yes,11,2026-05-27 10:40:53.195014-07:00
3,3,5dc5e1d3-7bf4-4ebb-ad7d-485ec1ea8703,9bd2b2b2-d42d-45e2-8d49-5d8e30f2bfbe,25-44,"[\n ""Another gender identity (like transgende...","Another gender identity (like transgender, non...","[\n ""White""\n]",White (only),active,published,...,Employee (non-management),Alameda County,Bay Area,Legal,It seems to be hitting creatives the hardest. ...,Prevent the concentration of wealth in the han...,"I've used it, and some savvy coworkers use it ...",Yes,11,2026-05-27 10:40:53.195014-07:00
4,4,0285d58b-2c7c-4c01-ad90-0282006e71a1,4603bf64-6d07-4176-a788-e2cf8dfa92d0,25-44,"[\n ""Woman""\n]",Woman (only),"[\n ""Hispanic or Latino""\n]",Hispanic or Latino (only),active,published,...,Manager,Los Angeles County,Los Angeles,Other,Right now it seems to be scaffolding the econo...,Build infrastructure to support long-term unem...,I believe that AI impacted the design/strategy...,Yes,11,2026-05-27 10:40:53.195014-07:00


In [4]:
OVERALL_AI_SENTIMENT_COLS = ["ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT"]
GOVERNMENT_ACTION_SENTIMENT_COLS = ["GOVERNMENT_ACTION_SUGGESTION"]

SENTIMENT_OPTIONS = ["PRO", "ANTI", "NEUTRAL"]

## Step 2: Zero-Shot Classification

In [5]:
MODEL = "claude-4-sonnet"

OVERALL_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI.
Classify the respondent's policy position and attitude toward AI as a technology — not the emotional tone of their writing.

PRO: The respondent views AI as broadly beneficial or more beneficial than harmful and supports its development, adoption, or expansion.
ANTI: The respondent views AI as broadly harmful or more harmful than good and opposes its development, adoption, or expansion.
NEUTRAL: The respondent is ambivalent, does not express a clear directional stance, or expresses views that are both strongly pro- or strongly anti-.

Respond with JSON only, no markdown: {"label": "pro|anti|neutral", "rationale": "one sentence"}
""".strip()

GOVERNMENT_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI governance.
Classify the respondent's policy position on government action regarding AI — not the emotional tone of their writing.

PRO: The respondent supports active government involvement (regulation, oversight, restrictions, public investment, or standards-setting).
ANTI: The respondent opposes government involvement and prefers a market-driven or hands-off approach.
NEUTRAL: The respondent is ambivalent, proposes a balanced approach, or does not express a clear directional stance.

Respond with JSON only, no markdown: {"label": "pro|anti|neutral", "rationale": "one sentence"}
""".strip()

In [6]:
def esc(s):
    # Escape for safe embedding in a Snowflake SQL string literal
    return s.replace("'", "''").replace("\n", " ").replace("\r", "")

sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    CASE
        WHEN COALESCE(ECONOMIC_IMPACT_EXPECTATION, '') != ''
          OR COALESCE(PERSONAL_AI_IMPACT, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content',
                    CONCAT(
                        'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                        ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                    )
                )
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS OVERALL_RAW,
    CASE
        WHEN COALESCE(GOVERNMENT_ACTION_SUGGESTION, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(GOVERNMENT_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content', COALESCE(GOVERNMENT_ACTION_SUGGESTION, ''))
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS GOVERNMENT_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
"""

cur = conn.cursor()
cur.execute(sql)
raw_df = cur.fetch_pandas_all()

In [7]:
raw_df.head()

,SURVEY_RESPONDENT_ID,OVERALL_RAW,GOVERNMENT_RAW
0,b2768fb1-bcc7-4350-875c-ec9f44a6c29d,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{..."
1,755de677-19c0-4522-a92b-c82472c3cd12,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{..."
2,71ccc7b9-9496-4327-bc78-39f32f48347e,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{..."
3,5dc5e1d3-7bf4-4ebb-ad7d-485ec1ea8703,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{..."
4,0285d58b-2c7c-4c01-ad90-0282006e71a1,"{\n ""choices"": [\n {\n ""messages"": ""{...","{\n ""choices"": [\n {\n ""messages"": ""{..."


In [8]:
def parse_cortex_response(raw):
    if raw is None:
        return pd.Series({"label": None, "rationale": None})
    try:
        content = json.loads(raw)["choices"][0]["messages"].strip()
        content = re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")
        # Model occasionally prefixes JSON with explanatory text — find where the object starts
        json_start = content.find("{")
        if json_start == -1:
            return pd.Series({"label": None, "rationale": None})
        result = json.loads(content[json_start:])
        return pd.Series({"label": result.get("label", "").lower() or None, "rationale": result.get("rationale", "")})
    except Exception as e:
        return pd.Series({"label": "error", "rationale": str(e)})

zero_shot_df = raw_df[["SURVEY_RESPONDENT_ID"]].copy()
zero_shot_df[["overall_ai_sentiment", "overall_rationale"]] = raw_df["OVERALL_RAW"].apply(parse_cortex_response)
zero_shot_df[["government_action_sentiment", "government_rationale"]] = raw_df["GOVERNMENT_RAW"].apply(parse_cortex_response)

zero_shot_df.head()

,SURVEY_RESPONDENT_ID,overall_ai_sentiment,overall_rationale,government_action_sentiment,government_rationale
0,b2768fb1-bcc7-4350-875c-ec9f44a6c29d,anti,The respondent views AI as leading to societal...,pro,The respondent explicitly calls for government...
1,755de677-19c0-4522-a92b-c82472c3cd12,anti,The respondent views AI as economically harmfu...,pro,The respondent explicitly calls for taxation a...
2,71ccc7b9-9496-4327-bc78-39f32f48347e,anti,The respondent views AI as leading to massive ...,pro,The respondent explicitly advocates for govern...
3,5dc5e1d3-7bf4-4ebb-ad7d-485ec1ea8703,neutral,The respondent acknowledges both negative impa...,pro,The respondent explicitly calls for government...
4,0285d58b-2c7c-4c01-ad90-0282006e71a1,neutral,The respondent describes AI's negative economi...,pro,The respondent advocates for expanded governme...


## Zeroshot Labelling Results

In [9]:
labeled_df = df.merge(zero_shot_df, on="SURVEY_RESPONDENT_ID", how="left")

audit_cols = [
    "SURVEY_RESPONDENT_ID",
    "ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT",
    "overall_ai_sentiment", "overall_rationale",
    "GOVERNMENT_ACTION_SUGGESTION",
    "government_action_sentiment", "government_rationale",
]
labeled_df[audit_cols].head(10)

,SURVEY_RESPONDENT_ID,ECONOMIC_IMPACT_EXPECTATION,PERSONAL_AI_IMPACT,overall_ai_sentiment,overall_rationale,GOVERNMENT_ACTION_SUGGESTION,government_action_sentiment,government_rationale
0,b2768fb1-bcc7-4350-875c-ec9f44a6c29d,It would be like the movie idiocracy. A societ...,There is an expectation that AI would be runni...,anti,The respondent views AI as leading to societal...,Narrowly define what AI can and cannot do.,pro,The respondent explicitly calls for government...
1,755de677-19c0-4522-a92b-c82472c3cd12,AI will make the economy worse. Nobody is actu...,Yes. AI has made the job market more difficult.,anti,The respondent views AI as economically harmfu...,Tax and regulate generative AI and LLM. They h...,pro,The respondent explicitly calls for taxation a...
2,71ccc7b9-9496-4327-bc78-39f32f48347e,AI will almost certainly lead to massive unemp...,Engineers have become much more productive at ...,anti,The respondent views AI as leading to massive ...,The government should nationalize critical ind...,pro,The respondent explicitly advocates for govern...
3,5dc5e1d3-7bf4-4ebb-ad7d-485ec1ea8703,It seems to be hitting creatives the hardest. ...,"I've used it, and some savvy coworkers use it ...",neutral,The respondent acknowledges both negative impa...,Prevent the concentration of wealth in the han...,pro,The respondent explicitly calls for government...
4,0285d58b-2c7c-4c01-ad90-0282006e71a1,Right now it seems to be scaffolding the econo...,I believe that AI impacted the design/strategy...,neutral,The respondent describes AI's negative economi...,Build infrastructure to support long-term unem...,pro,The respondent advocates for expanded governme...
5,a4df31f3-dea2-4417-a153-ee6e4deb8a71,"AI is expected to take over the workplace, but...",AI tools were rolled out at my former place of...,anti,The respondent predicts economic collapse due ...,AI is excellent at identifying patterns and pr...,pro,The respondent advocates for specific governme...
6,2cbb1d65-91b9-4695-80c5-601f7b61178f,"Increasingly, layoffs by large companies such ...","I'm in Marketing. AI skill, or at least ""AI fl...",neutral,The respondent acknowledges both significant c...,The government should absolutely pass regulati...,pro,The respondent explicitly calls for government...
7,7f19ae1c-bac9-4df0-84b3-fba9f0668b57,None,None,None,None,None,None,None
8,1af71ef7-d7cd-45ea-b808-c639715659a9,Unsure,It would help with my job if I had access to A...,pro,The respondent expresses a positive view of AI...,Unsure,neutral,The respondent explicitly states uncertainty a...
9,1f909d89-ef0b-40f3-b5bd-fe34249015f2,Destroying it!,"Yes, it has caused a slow down",anti,The respondent expresses clearly negative view...,Stop it!,neutral,The response is too brief and lacks context to...


In [10]:
print(labeled_df['overall_ai_sentiment'].value_counts(normalize=True))
print("\n")
print(labeled_df['government_action_sentiment'].value_counts(normalize=True))

overall_ai_sentiment
anti       0.584582
neutral    0.246253
pro        0.168451
error      0.000714
Name: proportion, dtype: float64


government_action_sentiment
pro        0.882396
neutral    0.072485
anti       0.045118
Name: proportion, dtype: float64


In [11]:
print(labeled_df['overall_ai_sentiment'].value_counts())
print(labeled_df['government_action_sentiment'].value_counts())

overall_ai_sentiment
anti       819
neutral    345
pro        236
error        1
Name: count, dtype: int64
government_action_sentiment
pro        1193
neutral      98
anti         61
Name: count, dtype: int64


In [12]:
for category in ["overall_ai_sentiment", "government_action_sentiment"]:
    print(f"=== {category} ===")
    print(labeled_df[category].value_counts(dropna=False))
    errors = labeled_df[labeled_df[category] == "error"]
    if len(errors):
        rationale_col = "overall_rationale" if category == "overall_ai_sentiment" else "government_rationale"
        print(f"\n{len(errors)} error(s):")
        print(errors[[rationale_col]].to_string())
    print()

=== overall_ai_sentiment ===
overall_ai_sentiment
anti       819
neutral    345
pro        236
None        79
error        1
Name: count, dtype: int64

1 error(s):
                                         overall_rationale
371  Expecting ',' delimiter: line 1 column 219 (char 218)

=== government_action_sentiment ===
government_action_sentiment
pro        1193
None        128
neutral      98
anti         61
Name: count, dtype: int64



## Step 3: Exemplar Selection

In [13]:
CHUNK_SIZE = 20
PICKS_PER_CHUNK = 5
STOP_THRESHOLD = 25
FINAL_TARGET = 10


def fmt_overall_text(row):
    parts = []
    if pd.notna(row["ECONOMIC_IMPACT_EXPECTATION"]) and row["ECONOMIC_IMPACT_EXPECTATION"]:
        parts.append(f"[Economic] {row['ECONOMIC_IMPACT_EXPECTATION'].strip()}")
    if pd.notna(row["PERSONAL_AI_IMPACT"]) and row["PERSONAL_AI_IMPACT"]:
        parts.append(f"[Personal] {row['PERSONAL_AI_IMPACT'].strip()}")
    return "\n".join(parts)


labeled_df["overall_text"] = labeled_df.apply(fmt_overall_text, axis=1)
labeled_df["government_text"] = labeled_df["GOVERNMENT_ACTION_SUGGESTION"].fillna("").str.strip()

In [14]:
EXEMPLAR_SYSTEM_PROMPT = """You are selecting canonical examples of a labeled survey response for use as few-shot classification examples.

The responses below are all labeled "{label}" for the category "{category}". Select the {n} best examples.

Prefer:
- Substantive over short or obvious: pick responses that make specific arguments or show a clear policy position
- Diverse: if possible, cover different angles of the "{label}" position rather than picking responses that repeat the same point

Return ONLY a JSON array of the response numbers you selected. Example: [2, 7, 14]"""


def cortex_complete(messages, max_tokens=150):
    # Parameterized binding lets the connector handle escaping — avoids Snowflake
    # interpreting \n in json.dumps output as a literal newline before PARSE_JSON sees it.
    messages_json = json.dumps(messages)
    sql = """SELECT SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        PARSE_JSON(%s),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', %s)
    )"""
    cur = conn.cursor()
    cur.execute(sql, [MODEL, messages_json, max_tokens])
    raw = cur.fetchone()[0]
    content = json.loads(raw)["choices"][0]["messages"].strip()
    return re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")


def select_from_chunk(texts, label, category, n_picks):
    numbered = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))
    system = EXEMPLAR_SYSTEM_PROMPT.format(label=label, category=category, n=n_picks)
    content = cortex_complete([
        {"role": "system", "content": system},
        {"role": "user", "content": numbered},
    ])
    try:
        picks = json.loads(content)
        return [p - 1 for p in picks if isinstance(p, int) and 1 <= p <= len(texts)]
    except Exception:
        print(f"  Warning: could not parse picks response: {content[:100]}")
        return list(range(min(n_picks, len(texts))))


def reduce_to_exemplars(texts, label, category):
    pool = list(range(len(texts)))

    while len(pool) > STOP_THRESHOLD:
        next_pool = []
        pool_texts = [texts[i] for i in pool]
        for start in range(0, len(pool_texts), CHUNK_SIZE):
            chunk = pool_texts[start:start + CHUNK_SIZE]
            picks = select_from_chunk(chunk, label, category, min(PICKS_PER_CHUNK, len(chunk)))
            next_pool.extend(pool[start + p] for p in picks)
        pool = next_pool
        print(f"  [{category} / {label}] pool → {len(pool)}")

    if len(pool) > FINAL_TARGET:
        pool_texts = [texts[i] for i in pool]
        picks = select_from_chunk(pool_texts, label, category, FINAL_TARGET)
        pool = [pool[p] for p in picks]

    return pool

In [15]:
CATEGORY_TEXT_COLS = {
    "overall_ai_sentiment": "overall_text",
    "government_action_sentiment": "government_text",
}

exemplar_dfs = {}

for category, text_col in CATEGORY_TEXT_COLS.items():
    exemplar_dfs[category] = {}
    for label in [s.lower() for s in SENTIMENT_OPTIONS]:
        mask = (labeled_df[category] == label) & (labeled_df[text_col] != "")
        subset = labeled_df[mask].copy()
        print(f"\n{category} / {label}: {len(subset)} responses")
        selected = reduce_to_exemplars(subset[text_col].tolist(), label, category)
        exemplar_dfs[category][label] = subset.iloc[selected].reset_index(drop=True)
        print(f"  → {len(selected)} exemplars selected")

print("\nDone!")


overall_ai_sentiment / pro: 236 responses
  [overall_ai_sentiment / pro] pool → 60
  [overall_ai_sentiment / pro] pool → 15
  → 10 exemplars selected

overall_ai_sentiment / anti: 819 responses
  [overall_ai_sentiment / anti] pool → 205
  [overall_ai_sentiment / anti] pool → 55
  [overall_ai_sentiment / anti] pool → 15
  → 10 exemplars selected

overall_ai_sentiment / neutral: 345 responses
  [overall_ai_sentiment / neutral] pool → 88
  [overall_ai_sentiment / neutral] pool → 25
  → 10 exemplars selected

government_action_sentiment / pro: 1193 responses
  [government_action_sentiment / pro] pool → 300
  [government_action_sentiment / pro] pool → 75
  [government_action_sentiment / pro] pool → 20
  → 10 exemplars selected

government_action_sentiment / anti: 61 responses
  [government_action_sentiment / anti] pool → 16
  → 10 exemplars selected

government_action_sentiment / neutral: 98 responses
  [government_action_sentiment / neutral] pool → 25
  → 10 exemplars selected

Done!


In [16]:
keep_cols = [
    "SURVEY_RESPONDENT_ID",
    "ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT",
    "GOVERNMENT_ACTION_SUGGESTION",
    "overall_ai_sentiment", "overall_rationale",
    "government_action_sentiment", "government_rationale",
]

parts = []
for category, text_col in CATEGORY_TEXT_COLS.items():
    for label, sub_df in exemplar_dfs[category].items():
        chunk = sub_df[keep_cols].copy()
        chunk["exemplar_category"] = category
        chunk["exemplar_label"] = label
        chunk["exemplar_text"] = sub_df[text_col].values
        parts.append(chunk)

exemplars_df = pd.concat(parts, ignore_index=True)
exemplars_df.to_csv("exemplars.csv", index=False)
print(f"Saved {len(exemplars_df)} rows to exemplars.csv")

Saved 60 rows to exemplars.csv


# Human In the Loop

Manually chosen examples for each label are fed back into the model for re-classification

In [17]:
example_lookup = {
    "overall_ai_sentiment": {
        "pro": [  # Survey response ids
            "51ce719b-e06f-41af-b9db-5286b655bf4f",
            "76ab3a2f-ae44-4459-88d3-70375f224b7e",
            "cbb89d9c-9dcc-44ba-9827-72e5d9c1d1e2",
            "557e0661-1d28-455a-b98e-f3e3dfd8e062",
            "d65b600a-5ba2-4b70-9141-df1ee129af21",
        ],
        "neutral": [
            "f34a115c-a529-47bb-9647-c4e83f02d13c",
            "dc1cb269-3b23-49b9-b298-6bf4aeb32d67",
            "f129ab70-82eb-4f99-9b11-fe0602717dae",
            "ef289e41-5f47-44c3-b946-930f1fe05406",
            "71d1b444-46c9-4194-b7c0-cf248a6a6c7b",
        ],
        "anti": [
            "0544a359-2705-4272-b30f-af60737c8988",
            "8e2c48bc-a2bd-463c-85a2-40ddec579075",
            "fe230ac7-f2e7-40eb-8583-76f4ad465acb",
            "e8a59dbb-908e-443e-98ed-8bfdd9f3c097",
            "6ae68715-e559-4f12-b1e4-87bced1253d3",
        ],
    },
    "government_action_sentiment": {
        "pro": [
            "20ddbdfa-2dbb-4b07-bfcb-54c791042e48",
            "9fe232b2-802d-4b9f-89c8-d1c17284682f",
            "f34a115c-a529-47bb-9647-c4e83f02d13c",
            "b0fd3ca5-0d52-44ed-9801-825730bbcccb",
            "fbe66a79-2117-44ef-b7f9-a7aa62b79ba7",
        ],
        "neutral": [
            "7106ed69-164a-41b0-ba55-76959995c243",
            "8c4f6546-eee8-4e0e-9b0a-f261a6a8cacb",
            "119a1dd0-9d4e-441d-9474-c8a74bf9c7db",
            "1fd6afb1-50e7-4c3a-a580-b4ae0ccacfcd",
            "9dd9031a-3635-4b04-b365-92dbdbd1cbc1",
        ],
        "anti": [
            "44296481-115b-41de-8f2a-feffefc24e46",
            "50c241f4-d09d-49d0-b490-c3a773cdc23f",
            "62ee1c31-1efb-4e5d-9ae2-ecc907c3bbfd",
            "ebffa2dc-12c9-4b55-9d18-e5ab4006b955",
            "a5d3241f-687a-4908-8737-85258f664193",
        ],
    }
}

## Step 4: Few-Shot Classification

In [18]:
def build_few_shot_section(category, text_col, rationale_col):
    lines = ["\n\nEXAMPLES (use these as calibration references):\n"]
    for label in ["pro", "anti", "neutral"]:
        ids = example_lookup[category][label]
        rows = labeled_df[labeled_df["SURVEY_RESPONDENT_ID"].isin(ids)]
        for _, row in rows.iterrows():
            text = row[text_col].strip().replace('"', "'")
            rationale = (row[rationale_col] or "").strip().replace('"', "'")
            lines.append(f'Response: "{text}"')
            lines.append(f'{{"label": "{label}", "rationale": "{rationale}"}}\n')
    return "\n".join(lines)

OVERALL_FEW_SHOT_PROMPT = OVERALL_SYSTEM_PROMPT + build_few_shot_section(
    "overall_ai_sentiment", "overall_text", "overall_rationale"
)
GOVERNMENT_FEW_SHOT_PROMPT = GOVERNMENT_SYSTEM_PROMPT + build_few_shot_section(
    "government_action_sentiment", "government_text", "government_rationale"
)

print(f"Overall prompt length: {len(OVERALL_FEW_SHOT_PROMPT)} chars")
print(f"Government prompt length: {len(GOVERNMENT_FEW_SHOT_PROMPT)} chars")
print("\n=== Overall few-shot prompt ===")
print(OVERALL_FEW_SHOT_PROMPT)
print("\n=== Government few-shot prompt ===")
print(GOVERNMENT_FEW_SHOT_PROMPT)

Overall prompt length: 27264 chars
Government prompt length: 13136 chars

=== Overall few-shot prompt ===
You are classifying California residents' survey responses about AI.
Classify the respondent's policy position and attitude toward AI as a technology — not the emotional tone of their writing.

PRO: The respondent views AI as broadly beneficial or more beneficial than harmful and supports its development, adoption, or expansion.
ANTI: The respondent views AI as broadly harmful or more harmful than good and opposes its development, adoption, or expansion.
NEUTRAL: The respondent is ambivalent, does not express a clear directional stance, or expresses views that are both strongly pro- or strongly anti-.

Respond with JSON only, no markdown: {"label": "pro|anti|neutral", "rationale": "one sentence"}

EXAMPLES (use these as calibration references):

Response: "[Economic] I expect large productivity gains as humans are able to scale their impact through agents especially in digital fiel

In [19]:
overall_sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    SNOWFLAKE.CORTEX.COMPLETE(
        '{MODEL}',
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_FEW_SHOT_PROMPT)}'),
            OBJECT_CONSTRUCT('role', 'user', 'content',
                CONCAT(
                    'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                    ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                )
            )
        ),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
    ) AS OVERALL_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
  AND (TRIM(COALESCE(ECONOMIC_IMPACT_EXPECTATION, '')) != ''
       OR TRIM(COALESCE(PERSONAL_AI_IMPACT, '')) != '')
"""

government_sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    SNOWFLAKE.CORTEX.COMPLETE(
        '{MODEL}',
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(GOVERNMENT_FEW_SHOT_PROMPT)}'),
            OBJECT_CONSTRUCT('role', 'user', 'content', GOVERNMENT_ACTION_SUGGESTION)
        ),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
    ) AS GOVERNMENT_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
  AND TRIM(COALESCE(GOVERNMENT_ACTION_SUGGESTION, '')) != ''
"""

cur = conn.cursor()

cur.execute(overall_sql)
overall_raw_df = cur.fetch_pandas_all()

cur.execute(government_sql)
government_raw_df = cur.fetch_pandas_all()

print(f"Overall: {len(overall_raw_df)} rows classified")
print(f"Government: {len(government_raw_df)} rows classified")

Overall: 1401 rows classified
Government: 1352 rows classified


In [20]:
overall_raw_df[["overall_ai_sentiment", "overall_rationale"]] = overall_raw_df["OVERALL_RAW"].apply(parse_cortex_response)
government_raw_df[["government_action_sentiment", "government_rationale"]] = government_raw_df["GOVERNMENT_RAW"].apply(parse_cortex_response)

few_shot_labeled_df = (
    df
    .merge(overall_raw_df[["SURVEY_RESPONDENT_ID", "overall_ai_sentiment", "overall_rationale"]], on="SURVEY_RESPONDENT_ID", how="left")
    .merge(government_raw_df[["SURVEY_RESPONDENT_ID", "government_action_sentiment", "government_rationale"]], on="SURVEY_RESPONDENT_ID", how="left")
)

few_shot_labeled_df[audit_cols].head(10)

,SURVEY_RESPONDENT_ID,ECONOMIC_IMPACT_EXPECTATION,PERSONAL_AI_IMPACT,overall_ai_sentiment,overall_rationale,GOVERNMENT_ACTION_SUGGESTION,government_action_sentiment,government_rationale
0,b2768fb1-bcc7-4350-875c-ec9f44a6c29d,It would be like the movie idiocracy. A societ...,There is an expectation that AI would be runni...,anti,The respondent views AI as leading to societal...,Narrowly define what AI can and cannot do.,pro,The respondent advocates for active government...
1,755de677-19c0-4522-a92b-c82472c3cd12,AI will make the economy worse. Nobody is actu...,Yes. AI has made the job market more difficult.,anti,The respondent views AI as making the economy ...,Tax and regulate generative AI and LLM. They h...,pro,The respondent explicitly calls for taxing and...
2,71ccc7b9-9496-4327-bc78-39f32f48347e,AI will almost certainly lead to massive unemp...,Engineers have become much more productive at ...,anti,The respondent views AI as leading to massive ...,The government should nationalize critical ind...,pro,The respondent advocates for extensive governm...
3,5dc5e1d3-7bf4-4ebb-ad7d-485ec1ea8703,It seems to be hitting creatives the hardest. ...,"I've used it, and some savvy coworkers use it ...",neutral,The respondent acknowledges AI's current negat...,Prevent the concentration of wealth in the han...,pro,The respondent advocates for active government...
4,0285d58b-2c7c-4c01-ad90-0282006e71a1,Right now it seems to be scaffolding the econo...,I believe that AI impacted the design/strategy...,anti,The respondent views AI as economically destab...,Build infrastructure to support long-term unem...,pro,The respondent advocates for active government...
5,a4df31f3-dea2-4417-a153-ee6e4deb8a71,"AI is expected to take over the workplace, but...",AI tools were rolled out at my former place of...,anti,The respondent predicts an imminent economic c...,AI is excellent at identifying patterns and pr...,pro,The respondent advocates for specific governme...
6,2cbb1d65-91b9-4695-80c5-601f7b61178f,"Increasingly, layoffs by large companies such ...","I'm in Marketing. AI skill, or at least ""AI fl...",anti,"The respondent views AI as primarily harmful, ...",The government should absolutely pass regulati...,pro,The respondent explicitly advocates for govern...
7,7f19ae1c-bac9-4df0-84b3-fba9f0668b57,None,None,NaN,NaN,None,NaN,NaN
8,1af71ef7-d7cd-45ea-b808-c639715659a9,Unsure,It would help with my job if I had access to A...,pro,The respondent expresses a desire to use AI to...,Unsure,neutral,The respondent explicitly states uncertainty w...
9,1f909d89-ef0b-40f3-b5bd-fe34249015f2,Destroying it!,"Yes, it has caused a slow down",anti,The respondent views AI as 'destroying' the ec...,Stop it!,anti,The respondent expresses opposition to current...


In [21]:
rationale_col = {
    "overall_ai_sentiment": "overall_rationale",
    "government_action_sentiment": "government_rationale",
}

for raw_df, sentiment_col, raw_col in [
    (overall_raw_df, "overall_ai_sentiment", "OVERALL_RAW"),
    (government_raw_df, "government_action_sentiment", "GOVERNMENT_RAW"),
]:
    errors = raw_df[raw_df[sentiment_col] == "error"]
    if errors.empty:
        print(f"No errors in {sentiment_col}")
        continue
    print(f"=== {sentiment_col}: {len(errors)} error(s) ===")
    for _, row in errors.iterrows():
        print(f"\nSURVEY_RESPONDENT_ID: {row['SURVEY_RESPONDENT_ID']}")
        print(f"Exception: {row[rationale_col[sentiment_col]]}")
        print(f"Raw response:\n{row[raw_col]}")
        print("---")

No errors in overall_ai_sentiment
No errors in government_action_sentiment


## Few-Shot vs Zero-Shot Comparison

In [22]:
for category in ["overall_ai_sentiment", "government_action_sentiment"]:
    print(f"=== {category} ===")

    zero = labeled_df[category].value_counts()
    few = few_shot_labeled_df[category].value_counts()
    comparison = pd.DataFrame({"zero_shot": zero, "few_shot": few}).fillna(0).astype(int)
    comparison["delta"] = comparison["few_shot"] - comparison["zero_shot"]
    print(comparison.to_string())

=== overall_ai_sentiment ===
                      zero_shot  few_shot  delta
overall_ai_sentiment                            
anti                        819       780    -39
error                         1         0     -1
neutral                     345       434     89
pro                         236       187    -49
=== government_action_sentiment ===
                             zero_shot  few_shot  delta
government_action_sentiment                            
pro                               1193      1127    -66
neutral                             98       142     44
anti                                61        83     22


## Final DBT Query

In [23]:
dbt_sql = f"""
WITH classified AS (
    SELECT
        SURVEY_RESPONDENT_ID,
        CASE
            WHEN TRIM(COALESCE(ECONOMIC_IMPACT_EXPECTATION, '')) != ''
              OR TRIM(COALESCE(PERSONAL_AI_IMPACT, '')) != ''
            THEN SNOWFLAKE.CORTEX.COMPLETE(
                '{MODEL}',
                ARRAY_CONSTRUCT(
                    OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_FEW_SHOT_PROMPT)}'),
                    OBJECT_CONSTRUCT('role', 'user', 'content',
                        CONCAT(
                            'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                            ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                        )
                    )
                ),
                OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
            )
        END AS overall_raw,
        CASE
            WHEN TRIM(COALESCE(GOVERNMENT_ACTION_SUGGESTION, '')) != ''
            THEN SNOWFLAKE.CORTEX.COMPLETE(
                '{MODEL}',
                ARRAY_CONSTRUCT(
                    OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(GOVERNMENT_FEW_SHOT_PROMPT)}'),
                    OBJECT_CONSTRUCT('role', 'user', 'content', GOVERNMENT_ACTION_SUGGESTION)
                ),
                OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
            )
        END AS government_raw
    FROM {{{{ ref('govocal_ai_survey_respondents') }}}}
    WHERE PUBLICATION_STATUS = 'published'
)

SELECT
    SURVEY_RESPONDENT_ID,
    TRY_PARSE_JSON(
        TRY_PARSE_JSON(overall_raw):choices[0]:messages::VARCHAR
    ):label::VARCHAR AS overall_ai_sentiment_classification,
    TRY_PARSE_JSON(
        TRY_PARSE_JSON(government_raw):choices[0]:messages::VARCHAR
    ):label::VARCHAR AS government_action_sentiment_classification
FROM classified
"""

print(dbt_sql)


WITH classified AS (
    SELECT
        SURVEY_RESPONDENT_ID,
        CASE
            WHEN TRIM(COALESCE(ECONOMIC_IMPACT_EXPECTATION, '')) != ''
              OR TRIM(COALESCE(PERSONAL_AI_IMPACT, '')) != ''
            THEN SNOWFLAKE.CORTEX.COMPLETE(
                'claude-4-sonnet',
                ARRAY_CONSTRUCT(
                    OBJECT_CONSTRUCT('role', 'system', 'content', 'You are classifying California residents'' survey responses about AI. Classify the respondent''s policy position and attitude toward AI as a technology — not the emotional tone of their writing.  PRO: The respondent views AI as broadly beneficial or more beneficial than harmful and supports its development, adoption, or expansion. ANTI: The respondent views AI as broadly harmful or more harmful than good and opposes its development, adoption, or expansion. NEUTRAL: The respondent is ambivalent, does not express a clear directional stance, or expresses views that are both strongly pro- or strongly anti-.  